In [ ]:
#getting BA/FA for all historic years required for the data structure
            
eu_msw_his = eu_msw.loc[eu_msw["TIME_PERIOD"].isin(np.arange(2010,2022))]
eu_msw_his= eu_msw_his[['GEN','DSP_I_RCV_E','TIME_PERIOD','unit','Country']]
eu_msw_his = eu_msw_his.rename(columns = {'GEN':"MSW_Gen", "DSP_I_RCV_E":"MSW_WasteInc"})  #this step is not needed
#Estimating ash quantities

<br>
From BAT document:<br>
    BA = 150-350 kg/t -> 15-35 % wt/wt<br>
    Boiler ash = 20-40 kg/t -> 2-4% wt/wt<br>
    FA = 15-60 -> 1.5-6 % wt/wt<br>
    <br>
    <br>
    CODES:<br>
        SLASH_flyAshesWasteInc<br>
        SLASH_bottomAshesWasteInc<br>

In [ ]:
eu_msw_his["SLASH_bottomAshesWasteInc"] = eu_msw_his['MSW_WasteInc']*0.28
eu_msw_his["SLASH_flyAshesWasteInc"] = eu_msw_his['MSW_WasteInc']*0.03

eu_msw_his.drop(['MSW_Gen','MSW_WasteInc'], axis = 'columns', inplace = True)

In [ ]:
# creating output data structure
columns = ['Waste Stream', 'Country', 'Year', 'Scenario', 'Substance main parent',
           'Stock/Flow ID', 'Value', 'Unit', 'Data Quality', 'Reference', 'Remark 1',
           'Remark 2', 'Remark 3']
eu_msw_his= pd.melt(eu_msw_his, id_vars=['Country','unit','TIME_PERIOD'], value_vars = ['SLASH_bottomAshesWasteInc',
                                'SLASH_flyAshesWasteInc'],var_name='Stock/Flow ID', value_name='Value')

In [ ]:
#Changing all values to tonnes
eu_msw_his["Value"] *=1000
eu_msw_his['unit'] = 't'
eu_msw_his = eu_msw_his.rename(columns = {'unit':"Unit"})
#Adding year, waste stream
eu_msw_his = eu_msw_his.rename(columns = {'TIME_PERIOD':'Year'})
eu_msw_his['Waste Stream'] = 'SLASH'
#Adding LowKey codes
subs_main_parent = {'SLASH_flyAshesWasteInc':'19 01 13','SLASH_bottomAshesWasteInc':'19 01 11',
                    'SLASH_boilerAshesWasteInc':'19 01 15'}
eu_msw_his['Substance main parent'] = eu_msw_his['Stock/Flow ID'].map(subs_main_parent)
#Rearrange columns

In [ ]:
eu_msw_his[['Scenario']] = 'OBS'
eu_msw_his[['Reference']] = np.nan
eu_msw_his[['Remark 2']] = np.nan
eu_msw_his.loc[eu_msw_his["Country"]=='GBR',['Remark 1']] = 'Estimated from OECD MSW incineration quantities'
conditions = (((eu_msw_his["Country"]=='DNK')&(eu_msw_his["Year"]==2010)) | ((eu_msw_his["Country"]=='IRL')&(eu_msw_his["Year"]==2013)) | ((eu_msw_his["Country"]=='IRL')&(eu_msw_his["Year"]==2015)) | ((eu_msw_his["Country"]=='ISL')&(eu_msw_his["Year"]==2019)))
eu_msw_his.loc[conditions,['Remark 1']]='Missing data, estimated from Eurostat MSW incineration quantities of neighbouring years '
eu_msw_his.loc[(eu_msw_his["Country"]!='GBR')& ~conditions,['Remark 1']]='Estimated from Eurostat MSW incineration quantities'
eu_msw_his[['Remark 3']] = 'Sowmya Ravisandiran'
eu_msw_his[['Data Quality']] = 2
eu_msw_his = eu_msw_his[columns]
eu_msw_his.drop(eu_msw_his.loc[eu_msw_his["Country"]=='EU27_2020'].index, inplace = True)
eu_msw_his.to_csv('Data_Structure_Task4.1_Task4.2.csv')